# Limits of Machine Learning

The concept of p-hacking describes how one can deliberately change the evaluation of one's data until the model achieves very high quality without any substantial explanation for it.
This concept originates from statistics, where a previously set p-value is meant to be undercut, supposedly proving particularly high quality of the model.
In machine learning, other metrics, such as accuracy, are used for measurement – a high accuracy suggests that the model has high prediction quality.
Therefore, strictly speaking, we are dealing with accuracy hacking here.
But is it really good to rely solely on one metric (p-value, accuracy, ...)?

To achieve particularly high accuracy values, one might exploit spurious correlations or other non-linear random relationships.
When enough attributes are considered simultaneously, there is a high probability that an attribute will show a relationship with the target variable.
In that case, the models work only with the existing dataset.
However, because it is just a random relationship between two attributes, it likely won't repeat later in production.
Therefore, it is always important to critically examine which data set would predict particular results.

Such behavior is a problem for all areas:
In science, false assumptions and predictive models may be used.
In practice, this means that potentially defective predictive models are put into operation.
As soon as someone relies on the predictions of the flawed model, it can lead to serious harm to people or the environment.
Financial losses can also occur.


In [4]:
import random

import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import plot_tree

As part of the initiatlization, here we set the visualiation backend of matplotlib:

In [5]:
%matplotlib ipympl

In [6]:
matplotlib.get_backend()

'ipympl'

First, a dataset with random attributes is generated.

In [7]:
# Fix random generator
random.seed(0)

number_rows = 50

df = pd.DataFrame({
    "Outlook": [random.choice(["Sunny", "Overcast", "Rain"]) for _ in range(number_rows)],
    "Temperature": [random.choice(["Hot", "Mild", "Cool"]) for _ in range(number_rows)],
    "Humidity": [random.choice(["High", "Normal"]) for _ in range(number_rows)],
    "Wind": [random.choice(["Weak", "Strong"]) for _ in range(number_rows)],
    "Play Tennis?": [random.choice(["Yes", "No"]) for _ in range(number_rows)],
})

df

,Outlook,Temperature,Humidity,Wind,Play Tennis?
0,Overcast,Cool,Normal,Weak,No
1,Overcast,Cool,Normal,Strong,No
2,Sunny,Hot,High,Weak,No
3,Overcast,Cool,High,Weak,No
4,Rain,Mild,High,Weak,Yes
5,Overcast,Mild,High,Strong,No
6,Overcast,Hot,High,Weak,No
7,Overcast,Cool,High,Strong,No
8,Overcast,Mild,Normal,Weak,Yes
9,Overcast,Cool,Normal,Weak,Yes


PS: This dataset looks very similar to a famous dataset used to introduce decision trees.
As I said, this is a completely randomly generated dataset.

For further use within algorithms, the nominal or ordinal scaled variables are converted into appropriate numerical representations.

In [8]:
df_cat = df.assign(**{
    col: df[col].astype('category').cat.codes for col in ["Humidity", "Wind", "Play Tennis?"]
})
df_cat = df_cat.assign(
    Outlook=df["Outlook"].astype(
        pd.CategoricalDtype(categories=["Rain", "Overcast", "Sunny"], ordered=True)).cat.codes,
    Temperature=df["Temperature"].astype(
        pd.CategoricalDtype(categories=["Cool", "Mild", "Hot"], ordered=True)).cat.codes
)
df_cat

,Outlook,Temperature,Humidity,Wind,Play Tennis?
0,1,0,1,1,0
1,1,0,1,0,0
2,2,2,0,1,0
3,1,0,0,1,0
4,0,1,0,1,1
5,1,1,0,0,0
6,1,2,0,1,0
7,1,0,0,0,0
8,1,1,1,1,1
9,1,0,1,1,1


Now let's see whether the decision tree finds a random connection and whether the cross-validation result still looks good.

In [9]:
dt = DecisionTreeClassifier()
eingabe = df_cat[list(set(df_cat.columns) - set(["Play Tennis?"]))]

ziel = df["Play Tennis?"]
X_train, X_test, y_train, y_test = train_test_split(eingabe, ziel, random_state=42)
dt.fit(X_train, y_train)
dt.score(X_test, y_test)

0.46153846153846156

<span style="color:blue; font-weight:bold">Task 1</span>

Interpret the numerical value.
In what unit is it?
Is this a spurious correlation?
Is this result plausible?
Why or why not?

*Answer*: ...

## Feature engineering done wrong

Feature engineering usually helps to generate useful numerical values.
For example, it makes sense to generate the time span instead of a start and end point, or to calculate the area instead of a length and a width.
But there are also cases in which project managers have added, multiplied, etc. arbitrary attributes without there being any substantive justification for doing so.
That's exactly what we're going to take to the extreme now.

Now we're going to add a lot of new attributes so that we can find at least one with which the decision tree can produce a good result.

In [ ]:
# The additions of two attributes are stored here
new_columns = {}

for column_A in list(set(df_cat.columns) - set(["Play Tennis?"])):
    for column_B in list(set(df_cat.columns) - set(["Play Tennis?"])):
        if column_A == column_B:
            continue

        addition = df_cat[column_A] + df_cat[column_B]
        addition.name = f"{column_A} + {column_B}"
        new_columns.update({addition.name: addition})

        subtraction = df_cat[column_A] - df_cat[column_B]
        subtraction.name = f"{column_A} - {column_B}"
        new_columns.update({subtraction.name: subtraction})

        multiplication = df_cat[column_A] * df_cat[column_B]
        multiplication.name = f"{column_A} * {column_B}"
        new_columns.update({multiplication.name: multiplication})


We'll now add these created attributes to the DataFrame.

In [ ]:
df_extended = df_cat.assign(**new_columns)

df_extended

Now let's see if the decision tree can work better with these generated features:

In [ ]:
target = df_extended["Play Tennis?"]
_input = df_extended[list(set(df_extended.columns) - set(["Play Tennis?"]))]

X_train, X_test, y_train, y_test = train_test_split(_input, target, random_state=74)
dt = DecisionTreeClassifier(random_state=5)
dt.fit(X_train, y_train)
dt.score(X_test, y_test)

<span style="color:blue; font-weight:bold">Task 2</span>

Interpret the numerical value.
What unit is it in?
Is this a spurious correlation?
Is this result plausible?
Why or why not?

*Answer*: ...

Let's have another look at the decision tree:

In [ ]:
plot_tree(
    dt,
    feature_names=df_extended.columns
)
plt.show()

<span style="color:blue; font-weight:bold">Task 3</span>

Please interpret what you see.
Which 5 attributes are most important in the decision making process?

*Answer*: ...

<span style="color:blue; font-weight:bold">Task 4</span>

Imagine you are 5 years in the future.
You work in customer service for a logistics company.
Due to a module you took in your master's degree, you are sure that you can now partially automate the processing of the emails that arrive every day.
However, you also know that you cannot handle this task yourself.
You are too busy with day-to-day business.
That is why you hire a service provider to take on this task.
You give the service provider the data that is required for the tasks.

After a long processing time, this service provider presents you with a solution.
Would you use a system whose "score" (calculated with `dt.score(X_test, y_test)`) is around $0.8$?
Give reasons.

*Answer*: ...

<span style="color:blue; font-weight:bold">Task 5</span>

If you requested improvements in the previous task, now assume that these have been made.

You would now like to understand whether the service provider's ML solution is actually ready for practical use.
You don't really trust the service provider.
You believe that the service provider has built an ML solution that would only work with the data you have already provided as part of the project.

What measures can you take to ensure that you are not being duped?
Assume that you do not have access to the source code
(this is the case, for example, if only an EXE file is provided or a web service is offered).
Present your concept.

*Answer*: ...

<span style="color:blue; font-weight:bold">Ungraded additional task</span>

What other mathematical operations besides addition and multiplication could you use in feature engineering?
Copy the Jupyter Notebook and add more features to `df_extended`.
Write down your observations.

*Answer*: ...

<a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Creative Commons License" style="border-width:0; display:inline" src="https://i.creativecommons.org/l/by/4.0/88x31.png" /></a> &nbsp;&nbsp;&nbsp;&nbsp;This work by Marvin Kastner is licensed under a <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Creative Commons Attribution 4.0 International License</a>.